# UC53564_41 — NLP Pipeline on Sentiment140
Roll No 1, Roll No 2

Self-contained Colab notebook: dataset profiling, normalization, N-gram LMs,
HMM POS tagging, CFG/CKY + dependency parsing, lexical semantics/WSD, and
vector semantics. **Random seed = 42** everywhere. Every function is defined
in this notebook — no external files to upload.

Run cells top to bottom.

## 0. Setup

In [ ]:
!pip -q install kagglehub nltk scikit-learn gensim spacy tabulate seaborn > /dev/null
!python -m spacy download en_core_web_sm -q > /dev/null


In [ ]:
import warnings, os, math, random, re, unicodedata
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
import spacy

warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)

RANDOM_SEED = 42

def set_global_seed(seed: int = RANDOM_SEED) -> None:
    """Seed random/numpy/PYTHONHASHSEED for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_global_seed(RANDOM_SEED)
print('Global seed fixed at', RANDOM_SEED)


In [ ]:
for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4',
            'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng',
            'treebank', 'universal_tagset']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(pkg, 'skip:', e)

nlp = spacy.load('en_core_web_sm')


### Edge-case utility functions
Used throughout every stage: empty input, non-ASCII, very long input, duplicates.

In [ ]:
MAX_REASONABLE_TOKENS = 5000

def is_empty_or_whitespace(text: str) -> bool:
    return text is None or len(str(text).strip()) == 0

def is_non_ascii(text: str) -> bool:
    return any(ord(ch) > 127 for ch in str(text))

def strip_non_ascii(text: str) -> str:
    normalized = unicodedata.normalize('NFKD', str(text))
    return normalized.encode('ascii', 'ignore').decode('ascii')

def truncate_long_input(tokens: List[str], max_tokens: int = MAX_REASONABLE_TOKENS) -> List[str]:
    return tokens[:max_tokens] if len(tokens) > max_tokens else tokens

def deduplicate_preserve_order(items: List[str]) -> List[str]:
    seen, out = set(), []
    for it in items:
        if it not in seen:
            seen.add(it); out.append(it)
    return out

PUNCT_RE = re.compile(r'^[\W_]+$', re.UNICODE)

def is_punctuation_token(token: str) -> bool:
    return bool(PUNCT_RE.match(token))

def normalize_case(token: str) -> str:
    return token.casefold()

def safe_tokenize(text: str, tokenizer) -> List[str]:
    """Guard tokenization against empty/whitespace/None input and very long input."""
    if is_empty_or_whitespace(text):
        return []
    tokens = tokenizer(text)
    return truncate_long_input(tokens)


## Stage i — Dataset identification, download, split (5 marks)
Dataset: **Sentiment140** (Kaggle `kazanova/sentiment140`), 1.6M tweets,
2 classes (0=negative, 4=positive), auto-labelled via emoticons.
We stratified-subsample to `SAMPLE_SIZE` tweets for Colab-feasible runtime
(stated explicitly) and split 80/10/10 with seed 42.

In [ ]:
COLUMN_NAMES = ['target', 'ids', 'date', 'flag', 'user', 'text']
SAMPLE_SIZE = 60_000

def download_sentiment140(use_kagglehub: bool = True) -> str:
    """Download Sentiment140 and return the path to the extracted CSV."""
    if use_kagglehub:
        import kagglehub
        path = kagglehub.dataset_download('kazanova/sentiment140')
        for f in os.listdir(path):
            if f.endswith('.csv'):
                return os.path.join(path, f)
        raise FileNotFoundError(f'No CSV found in {path}')
    else:
        import subprocess
        subprocess.run(['kaggle', 'datasets', 'download', '-d', 'kazanova/sentiment140',
                         '--unzip', '-p', './data'], check=True)
        return './data/training.1600000.processed.noemoticon.csv'

def load_raw(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding='latin-1', names=COLUMN_NAMES, header=None)
    df = df[~df['text'].apply(is_empty_or_whitespace)].reset_index(drop=True)
    return df

def describe_dataset(df: pd.DataFrame) -> dict:
    n_records = len(df)
    class_counts = df['target'].value_counts().sort_index()
    class_dist = (class_counts / n_records * 100).round(2)
    return {'n_records': n_records, 'n_classes': df['target'].nunique(),
            'class_counts': class_counts.to_dict(),
            'class_distribution_pct': class_dist.to_dict()}

def stratified_subsample(df: pd.DataFrame, sample_size: int = SAMPLE_SIZE, seed: int = RANDOM_SEED) -> pd.DataFrame:
    if sample_size >= len(df):
        return df.reset_index(drop=True)
    frac = sample_size / len(df)
    sampled = df.groupby('target', group_keys=False).apply(lambda g: g.sample(frac=frac, random_state=seed))
    return sampled.reset_index(drop=True)

def make_splits(df: pd.DataFrame, seed: int = RANDOM_SEED):
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=seed, stratify=df['target'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=seed, stratify=temp_df['target'])
    return (train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True))

def split_summary(train_df, val_df, test_df) -> pd.DataFrame:
    rows = []
    for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
        counts = part['target'].value_counts().sort_index()
        rows.append({'split': name, 'n': len(part),
                     **{f'class_{k}_pct': round(v / len(part) * 100, 2) for k, v in counts.items()}})
    return pd.DataFrame(rows)


In [ ]:
# --- Kaggle auth (uncomment if kagglehub prompts for it) ---
# from google.colab import files
# files.upload()  # select kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

csv_path = download_sentiment140(use_kagglehub=True)
print('Downloaded to:', csv_path)


In [ ]:
raw_df = load_raw(csv_path)
report = describe_dataset(raw_df)
print('Full dataset (before subsampling):')
for k, v in report.items():
    print(f'  {k}: {v}')

df = stratified_subsample(raw_df, sample_size=SAMPLE_SIZE, seed=RANDOM_SEED)
sub_report = describe_dataset(df)
print('\nWorking subsample used for the rest of the pipeline:')
for k, v in sub_report.items():
    print(f'  {k}: {v}')


In [ ]:
train_df, val_df, test_df = make_splits(df, seed=RANDOM_SEED)
summary_table = split_summary(train_df, val_df, test_df)
summary_table  # --> paste into report (Stage i)


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
pd.Series(sub_report['class_counts']).plot(kind='bar', ax=ax, color=['#c0392b', '#2980b9'])
ax.set_title('Class distribution (subsample)')
ax.set_xlabel('target'); ax.set_ylabel('count')
plt.tight_layout()
plt.savefig('outputs/stage1_class_distribution.png', dpi=150)
plt.show()


## Stage ii — Text normalization with NLTK (15 marks)
Tokenization -> case folding -> stopword removal -> stemming (Porter) &
lemmatization (WordNet, POS-aware) -> vocabulary cutoff + `<UNK>` policy.

In [ ]:
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

UNK_TOKEN = '<UNK>'
MIN_FREQ = 5

_tweet_tokenizer = TweetTokenizer(preserve_case=True, reduce_len=True, strip_handles=False)
_stemmer = PorterStemmer()
_lemmatizer = WordNetLemmatizer()
_stopword_set = set(stopwords.words('english'))

def tokenize(text: str) -> List[str]:
    return safe_tokenize(text, _tweet_tokenizer.tokenize)

def case_fold(tokens: List[str]) -> List[str]:
    return [normalize_case(t) for t in tokens]

def remove_stopwords(tokens: List[str], keep_negation: bool = True) -> List[str]:
    negators = {'not', 'no', 'never', "n't"}
    sw = _stopword_set - negators if keep_negation else _stopword_set
    return [t for t in tokens if t not in sw]

def stem_tokens(tokens: List[str]) -> List[str]:
    return [_stemmer.stem(t) for t in tokens]

def _wordnet_pos(nltk_pos: str) -> str:
    if nltk_pos.startswith('J'): return wordnet.ADJ
    if nltk_pos.startswith('V'): return wordnet.VERB
    if nltk_pos.startswith('N'): return wordnet.NOUN
    if nltk_pos.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

def lemmatize_tokens(tokens: List[str]) -> List[str]:
    if not tokens:
        return []
    tagged = nltk.pos_tag(tokens)
    return [_lemmatizer.lemmatize(tok, _wordnet_pos(pos)) for tok, pos in tagged]

def normalize_pipeline(text: str, do_stopwords: bool = True) -> dict:
    raw_tokens = tokenize(text)
    cased = case_fold(raw_tokens)
    no_sw = remove_stopwords(cased) if do_stopwords else cased
    stemmed = stem_tokens(no_sw)
    lemmatized = lemmatize_tokens(no_sw)
    return {'raw_tokens': raw_tokens, 'cased': cased, 'no_stopwords': no_sw,
            'stemmed': stemmed, 'lemmatized': lemmatized}

def stemmer_vs_lemmatizer_examples() -> List[dict]:
    """Two curated disagreement examples: 'better' (adj) and 'universities' (noun)."""
    examples = []
    for word, pos in [('better', 'a'), ('universities', 'n')]:
        stem = _stemmer.stem(word)
        lemma = _lemmatizer.lemmatize(word, pos)
        examples.append({'word': word, 'porter_stem': stem, 'wordnet_lemma': lemma, 'agree': stem == lemma})
    return examples

def build_vocab(token_lists: List[List[str]], min_freq: int = MIN_FREQ):
    freq = Counter(t for toks in token_lists for t in toks)
    vocab = {t for t, c in freq.items() if c >= min_freq}
    vocab.add(UNK_TOKEN)
    return freq, vocab

def apply_unk(tokens: List[str], vocab: set) -> List[str]:
    return [t if t in vocab else UNK_TOKEN for t in tokens]

def vocab_report(freq: Counter, vocab: set) -> dict:
    total_tokens = sum(freq.values())
    vocab_size = len(vocab)
    ttr = round(vocab_size / total_tokens, 4) if total_tokens else 0.0
    return {'vocabulary_size': vocab_size, 'token_count': total_tokens,
            'type_token_ratio': ttr, 'min_freq_cutoff': MIN_FREQ}


In [ ]:
sample_tweet = train_df['text'].iloc[0]
print('Example raw tweet:', sample_tweet)
pipeline_out = normalize_pipeline(sample_tweet)
for stage, toks in pipeline_out.items():
    print(f'{stage:15s}: {toks}')


In [ ]:
# Stemmer vs lemmatizer disagreement (2 curated examples)
examples = stemmer_vs_lemmatizer_examples()
pd.DataFrame(examples)


In [ ]:
train_texts = train_df['text'].tolist()
train_token_lists = [case_fold(tokenize(t)) for t in train_texts]
train_token_lists_nosw = [remove_stopwords(toks) for toks in train_token_lists]

freq, vocab = build_vocab(train_token_lists_nosw, min_freq=MIN_FREQ)
report_vocab = vocab_report(freq, vocab)
report_vocab


In [ ]:
example_toks = train_token_lists_nosw[1]
print('Before UNK:', example_toks)
print('After UNK :', apply_unk(example_toks, vocab))


**Edge cases checked for Stage ii:**

In [ ]:
edge_cases = ['', '   ', 'ok', '!!! ???', 'SoOoO GOOD!!', 'caf\u00e9 \U0001F600 na\u00efve',
              'good good good bad bad', 'word ' * 6000]
for text in edge_cases:
    toks = tokenize(text)
    label = text if len(text) < 40 else text[:40] + f'...(len={len(text)})'
    print(f'{label!r:55s} -> {len(toks)} tokens, empty={is_empty_or_whitespace(text)}, non_ascii={is_non_ascii(text)}')


## Stage iii — N-gram language models (20 marks)
Unigram/bigram/trigram counts with `<s>`/`</s>` boundaries; Laplace add-1,
add-k (k tuned on validation), and Kneser-Ney smoothing; test perplexity
for every combination; 5 sampled sentences.

In [ ]:
from nltk.lm import Laplace, Lidstone, KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.util import pad_sequence, ngrams as nltk_ngrams

CANDIDATE_K = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]

def sentences_to_token_lists(token_lists: List[List[str]]) -> List[List[str]]:
    return [t for t in token_lists if len(t) > 0]

def train_ngram_model(model_cls, order: int, train_sents: List[List[str]], **kwargs):
    train_data, vocab_iter = padded_everygram_pipeline(order, train_sents)
    model = model_cls(order, **kwargs) if kwargs else model_cls(order)
    model.fit(train_data, vocab_iter)
    return model

def perplexity_of(model, order: int, test_sents: List[List[str]]) -> float:
    all_ngrams = []
    for sent in test_sents:
        if len(sent) == 0:
            continue
        padded = list(pad_sequence(sent, order, pad_left=True, pad_right=True,
                                    left_pad_symbol='<s>', right_pad_symbol='</s>'))
        all_ngrams.extend(list(nltk_ngrams(padded, order)))
    if not all_ngrams:
        return float('inf')
    return model.perplexity(all_ngrams)

def tune_add_k(order, train_sents, val_sents, candidates=CANDIDATE_K):
    results = {}
    for k in candidates:
        model = train_ngram_model(Lidstone, order, train_sents, gamma=k)
        results[k] = perplexity_of(model, order, val_sents)
    best_k = min(results, key=results.get)
    return best_k, results

def sample_sentences(model, order: int, n_sentences: int = 5, max_len: int = 20, seed: int = RANDOM_SEED):
    rng = np.random.RandomState(seed)
    sentences = []
    for i in range(n_sentences):
        context = ['<s>'] * (order - 1) if order > 1 else []
        words = []
        for _ in range(max_len):
            w = model.generate(1, text_seed=context if context else None,
                                random_seed=int(rng.randint(0, 1_000_000)))
            if isinstance(w, list):
                w = w[0]
            if w == '</s>':
                break
            words.append(w)
            context = (context + [w])[-(order - 1):] if order > 1 else []
        sentences.append(' '.join(words) if words else '(empty generation)')
    return sentences

def run_all_ngram_experiments(train_sents, val_sents, test_sents) -> dict:
    results = {'perplexity': {}, 'best_k': {}, 'samples': {}}
    for order in [1, 2, 3]:
        laplace = train_ngram_model(Laplace, order, train_sents)
        results['perplexity'][(order, 'laplace_add1')] = perplexity_of(laplace, order, test_sents)

        best_k, val_ppls = tune_add_k(order, train_sents, val_sents)
        addk_model = train_ngram_model(Lidstone, order, train_sents, gamma=best_k)
        results['perplexity'][(order, f'add_k(k={best_k})')] = perplexity_of(addk_model, order, test_sents)
        results['best_k'][order] = best_k

        if order >= 2:
            kn = train_ngram_model(KneserNeyInterpolated, order, train_sents)
        else:
            kn = laplace
        results['perplexity'][(order, 'kneser_ney')] = perplexity_of(kn, order, test_sents)
        results['samples'][order] = sample_sentences(kn if order >= 2 else laplace, order, n_sentences=5)
    return results


In [ ]:
val_texts = val_df['text'].tolist()
test_texts = test_df['text'].tolist()

def to_sents(texts):
    return sentences_to_token_lists([case_fold(tokenize(t)) for t in texts])

train_sents = to_sents(train_texts)
val_sents = to_sents(val_texts)
test_sents = to_sents(test_texts)
print(len(train_sents), len(val_sents), len(test_sents), 'sentences (train/val/test)')


In [ ]:
# NOTE: adjust n_docs for a quick smoke test vs. the full report run.
n_docs = 4000
results = run_all_ngram_experiments(train_sents[:n_docs], val_sents[:1000], test_sents[:1000])

ppl_rows = [{'order': o, 'smoothing': s, 'test_perplexity': round(p, 3)}
            for (o, s), p in results['perplexity'].items()]
ppl_table = pd.DataFrame(ppl_rows).sort_values(['order', 'smoothing'])
ppl_table


In [ ]:
print('Best add-k per order (tuned on validation):', results['best_k'])
for order, sents in results['samples'].items():
    print(f'\n--- Order {order} sampled sentences ---')
    for s in sents:
        print(' ', s)


## Stage iv — Part-of-Speech tagging (20 marks)
HMM Viterbi tagger (from scratch) vs a most-frequent-tag baseline vs spaCy,
built from the NLTK Penn Treebank sample with the universal tagset. Unknown
words handled via suffix features.

In [ ]:
from nltk.corpus import treebank

START, END, SUFFIX_LEN = '<s>', '</s>', 3

def load_tagged_sentences(seed=RANDOM_SEED, test_size=0.2):
    sents = list(treebank.tagged_sents(tagset='universal'))
    return train_test_split(sents, test_size=test_size, random_state=seed)

class HMMTagger:
    """From-scratch bigram HMM POS tagger with Viterbi decoding and
    suffix-based unknown-word handling."""

    def __init__(self, suffix_len: int = SUFFIX_LEN):
        self.suffix_len = suffix_len
        self.tag_counts = Counter()
        self.transition = defaultdict(Counter)
        self.emission = defaultdict(Counter)
        self.suffix_tag_counts = defaultdict(Counter)
        self.vocab = set()
        self.tags = set()

    def fit(self, tagged_sents):
        word_freq = Counter(w.lower() for sent in tagged_sents for w, _ in sent)
        for sent in tagged_sents:
            prev_tag = START
            for word, tag in sent:
                word = word.lower()
                self.vocab.add(word); self.tags.add(tag)
                self.tag_counts[tag] += 1
                self.transition[prev_tag][tag] += 1
                self.emission[tag][word] += 1
                if word_freq[word] == 1:
                    suf = word[-self.suffix_len:] if len(word) >= self.suffix_len else word
                    self.suffix_tag_counts[suf][tag] += 1
                prev_tag = tag
            self.transition[prev_tag][END] += 1
        self.tags.add(END)
        self.tags_list = sorted(self.tags)

    def _trans_prob(self, prev_tag, tag):
        total = sum(self.transition[prev_tag].values())
        if total == 0:
            return 1e-8
        return (self.transition[prev_tag][tag] + 1e-3) / (total + 1e-3 * len(self.tags_list))

    def _emit_prob(self, tag, word):
        word = word.lower()
        total = sum(self.emission[tag].values())
        if word in self.vocab:
            return (self.emission[tag][word] + 1e-6) / (total + 1e-6 * len(self.vocab))
        suf = word[-self.suffix_len:] if len(word) >= self.suffix_len else word
        suf_counts = self.suffix_tag_counts.get(suf)
        if suf_counts:
            suf_total = sum(suf_counts.values())
            return (suf_counts[tag] + 1e-3) / (suf_total + 1e-3 * len(self.tags_list))
        return 1.0 / len(self.tags_list)

    def viterbi(self, words):
        if not words:
            return []
        n = len(words)
        tags = self.tags_list
        V = [{} for _ in range(n)]
        backptr = [{} for _ in range(n)]

        for t in tags:
            V[0][t] = math.log(self._trans_prob(START, t)) + math.log(self._emit_prob(t, words[0]))
            backptr[0][t] = None

        for i in range(1, n):
            for t in tags:
                best_score, best_prev = -math.inf, None
                emit = math.log(self._emit_prob(t, words[i]))
                for pt in tags:
                    score = V[i - 1][pt] + math.log(self._trans_prob(pt, t)) + emit
                    if score > best_score:
                        best_score, best_prev = score, pt
                V[i][t] = best_score
                backptr[i][t] = best_prev

        best_last = max(tags, key=lambda t: V[n - 1][t])
        seq = [best_last]
        for i in range(n - 1, 0, -1):
            seq.append(backptr[i][seq[-1]])
        seq.reverse()
        return seq

def most_frequent_tag_baseline(train_sents):
    word_tag_counts = defaultdict(Counter)
    tag_counts = Counter()
    for sent in train_sents:
        for w, t in sent:
            word_tag_counts[w.lower()][t] += 1
            tag_counts[t] += 1
    lookup = {w: c.most_common(1)[0][0] for w, c in word_tag_counts.items()}
    default_tag = tag_counts.most_common(1)[0][0]
    return lookup, default_tag

def apply_baseline(lookup, default_tag, words):
    return [lookup.get(w.lower(), default_tag) for w in words]

def tagset_map_spacy_to_universal(spacy_pos: str) -> str:
    mapping = {'ADJ': 'ADJ', 'ADP': 'ADP', 'ADV': 'ADV', 'AUX': 'VERB', 'CCONJ': 'CONJ',
               'DET': 'DET', 'INTJ': 'X', 'NOUN': 'NOUN', 'NUM': 'NUM', 'PART': 'PRT',
               'PRON': 'PRON', 'PROPN': 'NOUN', 'PUNCT': '.', 'SCONJ': 'ADP',
               'SYM': 'X', 'VERB': 'VERB', 'X': 'X', 'SPACE': 'X'}
    return mapping.get(spacy_pos, 'X')

def confusion_pairs(gold, pred, top_n=5):
    mism = Counter((g, p) for g, p in zip(gold, pred) if g != p)
    return mism.most_common(top_n)

def accuracy(gold, pred):
    if not gold:
        return 0.0
    return round(sum(1 for g, p in zip(gold, pred) if g == p) / len(gold), 4)


In [ ]:
pt_train, pt_test = load_tagged_sentences(seed=RANDOM_SEED)
print(len(pt_train), 'train sents,', len(pt_test), 'test sents (Treebank, universal tagset)')

hmm = HMMTagger()
hmm.fit(pt_train)
lookup, default_tag = most_frequent_tag_baseline(pt_train)


In [ ]:
gold_tags, hmm_pred, base_pred, spacy_pred = [], [], [], []

for sent in pt_test:
    words = [w for w, _ in sent]
    tags = [t for _, t in sent]
    gold_tags.extend(tags)
    hmm_pred.extend(hmm.viterbi(words))
    base_pred.extend(apply_baseline(lookup, default_tag, words))

    doc = nlp(' '.join(words))
    spacy_pred.extend([tagset_map_spacy_to_universal(tok.pos_) for tok in doc][:len(words)])

print('HMM Viterbi accuracy   :', accuracy(gold_tags, hmm_pred))
print('Most-frequent baseline :', accuracy(gold_tags, base_pred))
print('spaCy (mapped) accuracy:', accuracy(gold_tags, spacy_pred))


In [ ]:
top5_confused = confusion_pairs(gold_tags, hmm_pred, top_n=5)
pd.DataFrame(top5_confused, columns=['(gold, predicted)', 'count'])


## Stage v — Syntactic parsing (15 marks)
A 12+ rule CFG converted to CNF, CKY parsing with two trees for the
ambiguous sentence *"I saw the movie with friends"*, plus spaCy dependency
parses and subject-verb-object triple extraction.

In [ ]:
from nltk import CFG, Tree
from nltk.grammar import Nonterminal, Production

TOY_GRAMMAR = CFG.fromstring('''
    S -> NP VP
    NP -> Det N | 'I' | NP PP | N
    VP -> V NP | V PP | VP PP
    PP -> P NP
    Det -> 'the' | 'a' | 'my'
    N -> 'movie' | 'friends' | 'phone' | 'park' | 'day'
    V -> 'saw' | 'liked' | 'lost' | 'enjoyed'
    P -> 'with' | 'in' | 'at'
''')
# NOTE: NP -> N allows a bare noun (e.g. plural 'friends' with no determiner)
# to stand alone as an NP, which is what makes the PP-attachment ambiguity
# in "I saw the movie with friends" resolvable both ways (see below).
AMBIGUOUS_SENTENCE = 'I saw the movie with friends'.split()

def to_cnf(grammar: CFG) -> CFG:
    productions = list(grammar.productions())
    non_unit = [p for p in productions if not (len(p.rhs()) == 1 and isinstance(p.rhs()[0], Nonterminal))]
    unit = [p for p in productions if (len(p.rhs()) == 1 and isinstance(p.rhs()[0], Nonterminal))]
    expanded = list(non_unit)
    # Take the transitive closure of unit productions (A -> B -> C chains),
    # not just a single hop, so multi-step unit chains still collapse correctly.
    closure = set((p.lhs(), p.rhs()[0]) for p in unit)
    changed = True
    while changed:
        changed = False
        for a, b in list(closure):
            for c, d in list(closure):
                if b == c and (a, d) not in closure:
                    closure.add((a, d))
                    changed = True
    for lhs, target in closure:
        for p in non_unit:
            if p.lhs() == target:
                expanded.append(Production(lhs, p.rhs()))

    binarized = []
    counter = [0]
    def bin_name():
        counter[0] += 1
        return Nonterminal(f'X{counter[0]}')

    for p in expanded:
        rhs = list(p.rhs())
        lhs = p.lhs()
        if len(rhs) <= 2:
            binarized.append(Production(lhs, tuple(rhs)))
        else:
            cur_lhs = lhs
            while len(rhs) > 2:
                new_nt = bin_name()
                binarized.append(Production(cur_lhs, (rhs[0], new_nt)))
                cur_lhs = new_nt
                rhs = rhs[1:]
            binarized.append(Production(cur_lhs, tuple(rhs)))
    return CFG(grammar.start(), binarized)

def cky_parse(grammar: CFG, tokens: List[str]) -> List[Tree]:
    cnf = to_cnf(grammar)
    n = len(tokens)
    table = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n + 1)]
    lexical, binary = defaultdict(list), defaultdict(list)
    for p in cnf.productions():
        if len(p.rhs()) == 1:
            lexical[p.rhs()[0]].append(p.lhs())
        elif len(p.rhs()) == 2:
            binary[(p.rhs()[0], p.rhs()[1])].append(p.lhs())

    for i in range(n):
        word = tokens[i]
        for lhs in lexical.get(word, []):
            table[i][i + 1][lhs].append(Tree(str(lhs), [word]))

    for span in range(2, n + 1):
        for i in range(0, n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                for (B, C), lhss in binary.items():
                    if B in table[i][k] and C in table[k][j]:
                        for lhs in lhss:
                            for bt in table[i][k][B]:
                                for ct in table[k][j][C]:
                                    table[i][j][lhs].append(Tree(str(lhs), [bt, ct]))
    start = cnf.start()
    return table[0][n].get(start, [])

def spacy_dependency_table(nlp, sentences: List[str]) -> pd.DataFrame:
    rows = []
    for sent in sentences:
        if not sent.strip():
            continue
        doc = nlp(sent)
        for tok in doc:
            rows.append({'sentence': sent, 'token': tok.text, 'dep': tok.dep_,
                         'head': tok.head.text, 'pos': tok.pos_})
    return pd.DataFrame(rows)

def extract_svo_triples(nlp, sentences: List[str]):
    triples = []
    for sent in sentences:
        if not sent.strip():
            continue
        doc = nlp(sent)
        for tok in doc:
            if tok.pos_ == 'VERB' or tok.dep_ == 'ROOT':
                subj = [c.text for c in tok.children if c.dep_ in ('nsubj', 'nsubjpass')]
                obj = [c.text for c in tok.children if c.dep_ in ('dobj', 'obj', 'attr', 'pobj')]
                if subj and obj:
                    triples.append((subj[0], tok.text, obj[0]))
    return triples


In [ ]:
print('Number of CFG productions:', len(TOY_GRAMMAR.productions()))
trees = cky_parse(TOY_GRAMMAR, AMBIGUOUS_SENTENCE)
print(f'Found {len(trees)} parse(s) for: {" ".join(AMBIGUOUS_SENTENCE)}')
for i, t in enumerate(trees[:2]):
    print(f'\n--- Tree {i+1} ---')
    t.pretty_print()


In [ ]:
sample_sentences_for_dep = train_df['text'].sample(10, random_state=RANDOM_SEED).tolist()
dep_table = spacy_dependency_table(nlp, sample_sentences_for_dep)
dep_table.head(10)


In [ ]:
triples = extract_svo_triples(nlp, sample_sentences_for_dep)
pd.DataFrame(triples, columns=['subject', 'verb', 'object'])


## Stage vi — Lexical semantics & WSD (10 marks)
WordNet senses/hypernyms/hyponyms/antonyms for 5 polysemous words from the
data, plus a from-scratch simplified Lesk algorithm evaluated on 20
hand-labelled sentences.

In [ ]:
from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize

POLYSEMOUS_WORDS = ['bank', 'light', 'well', 'book', 'play']

def wordnet_profile(word: str, max_senses: int = 3) -> dict:
    synsets = wn.synsets(word)[:max_senses]
    profile = {'word': word, 'senses': []}
    for s in synsets:
        hypernyms = [h.lemma_names()[0] for h in s.hypernyms()[:3]]
        hyponyms = [h.lemma_names()[0] for h in s.hyponyms()[:3]]
        antonyms = []
        for lemma in s.lemmas():
            antonyms.extend([a.name() for a in lemma.antonyms()])
        profile['senses'].append({'synset': s.name(), 'definition': s.definition(),
                                   'hypernyms': hypernyms, 'hyponyms': hyponyms,
                                   'antonyms': list(set(antonyms))})
    return profile

def simplified_lesk(word: str, sentence: str) -> Optional[str]:
    context = set(w.lower() for w in word_tokenize(sentence)) if sentence.strip() else set()
    synsets = wn.synsets(word)
    if not synsets:
        return None
    if not context:
        return synsets[0].name()
    best_sense, best_overlap = synsets[0], -1
    for s in synsets:
        signature = set(word_tokenize(s.definition()))
        for ex in s.examples():
            signature |= set(word_tokenize(ex))
        overlap = len(signature & context)
        if overlap > best_overlap:
            best_overlap, best_sense = overlap, s
    return best_sense.name()

def evaluate_lesk(labelled_examples):
    results, correct = [], 0
    for word, sentence, gold in labelled_examples:
        pred = simplified_lesk(word, sentence)
        is_correct = pred == gold
        correct += int(is_correct)
        results.append({'word': word, 'sentence': sentence, 'gold': gold,
                         'predicted': pred, 'correct': is_correct})
    acc = round(correct / len(labelled_examples), 4) if labelled_examples else 0.0
    return {'accuracy': acc, 'details': results}


In [ ]:
for w in POLYSEMOUS_WORDS:
    profile = wordnet_profile(w)
    print(f'--- {w} ---')
    for sense in profile['senses']:
        print(' ', sense)
    print()


In [ ]:
# 20 hand-labelled (word, sentence, gold_synset) examples.
# Swap in sentences drawn from your own tweets; use wn.synsets(word) to
# confirm the correct gold synset name for each.
labelled_examples = [
    ('bank', 'I deposited money at the bank today', 'bank.n.01'),
    ('bank', 'We sat on the river bank and watched the sunset', 'bank.n.01'),
    ('light', 'Turn on the light, it is dark in here', 'light.n.01'),
    ('light', 'This bag is very light to carry', 'light.a.01'),
    ('well', 'She sang very well at the concert', 'well.r.01'),
    ('well', 'They dug a well to find water', 'well.n.01'),
    ('book', 'I am reading a great book right now', 'book.n.01'),
    ('book', 'Please book a table for two tonight', 'book.v.01'),
    ('play', 'The kids went out to play in the park', 'play.v.01'),
    ('play', 'We watched a play at the theatre', 'play.n.01'),
    ('bank', 'The plane began to bank sharply to the left', 'bank.v.01'),
    ('light', 'The sky turned light just before dawn', 'light.a.01'),
    ('well', 'I am not feeling well today', 'well.a.01'),
    ('book', 'The police wanted to book him for speeding', 'book.v.02'),
    ('play', 'He likes to play the guitar every evening', 'play.v.03'),
    ('bank', 'Do not bank on the weather being good', 'bank.v.02'),
    ('light', 'Can you light the candle for me', 'light.v.01'),
    ('well', 'The company performed well this quarter', 'well.r.01'),
    ('book', 'This is a very interesting book about history', 'book.n.01'),
    ('play', 'The children love to play video games', 'play.v.01'),
]
lesk_eval = evaluate_lesk(labelled_examples)
print('Simplified Lesk accuracy:', lesk_eval['accuracy'])
pd.DataFrame(lesk_eval['details'])


## Stage vii — Vector semantics (15 marks)
Term-document matrix, word-word co-occurrence matrix (window=4), TF-IDF,
PPMI, and a Gensim Word2Vec skip-gram model, compared via nearest-neighbour
cosine similarity for 5 query words.

In [ ]:
CONTEXT_WINDOW = 4

def term_document_matrix(documents: List[str]):
    vec = CountVectorizer()
    X = vec.fit_transform(documents)
    return X.T.tocsr(), vec.get_feature_names_out()

def tfidf_matrix(documents: List[str]):
    vec = TfidfVectorizer()
    X = vec.fit_transform(documents)
    return X, vec.get_feature_names_out()

def word_word_cooccurrence(tokenized_docs: List[List[str]], window: int = CONTEXT_WINDOW):
    vocab = sorted({w for doc in tokenized_docs for w in doc})
    idx = {w: i for i, w in enumerate(vocab)}
    n = len(vocab)
    M = lil_matrix((n, n), dtype=np.float64)
    for doc in tokenized_docs:
        for center_pos, center_word in enumerate(doc):
            start = max(0, center_pos - window)
            end = min(len(doc), center_pos + window + 1)
            for pos in range(start, end):
                if pos == center_pos:
                    continue
                M[idx[center_word], idx[doc[pos]]] += 1
    return M.tocsr(), vocab

def compute_ppmi(cooc_matrix, smoothing: float = 1e-8):
    M = cooc_matrix.toarray().astype(np.float64)
    total = M.sum() + smoothing
    row_sums = M.sum(axis=1, keepdims=True) + smoothing
    col_sums = M.sum(axis=0, keepdims=True) + smoothing
    expected = (row_sums @ col_sums) / total
    with np.errstate(divide='ignore', invalid='ignore'):
        pmi = np.log2((M + smoothing) / (expected + smoothing))
    pmi[np.isnan(pmi)] = 0.0
    return np.maximum(pmi, 0.0)

def train_word2vec(tokenized_docs, vector_size=100, window=CONTEXT_WINDOW, min_count=3, seed=RANDOM_SEED):
    from gensim.models import Word2Vec
    return Word2Vec(sentences=tokenized_docs, vector_size=vector_size, window=window,
                     min_count=min_count, sg=1, seed=seed, workers=1)

def nearest_neighbours_tfidf(query_word, terms, X_tfidf, top_n=5):
    terms = list(terms)
    if query_word not in terms:
        return []
    X = X_tfidf.T if X_tfidf.shape[0] != len(terms) else X_tfidf
    idx = terms.index(query_word)
    sims = cosine_similarity(X[idx], X).flatten()
    ranked = np.argsort(-sims)
    return [(terms[i], round(float(sims[i]), 4)) for i in ranked if i != idx][:top_n]

def nearest_neighbours_ppmi(query_word, vocab, ppmi_matrix, top_n=5):
    if query_word not in vocab:
        return []
    idx = vocab.index(query_word)
    sims = cosine_similarity(ppmi_matrix[idx:idx + 1], ppmi_matrix).flatten()
    ranked = np.argsort(-sims)
    return [(vocab[i], round(float(sims[i]), 4)) for i in ranked if i != idx][:top_n]

def nearest_neighbours_word2vec(query_word, w2v_model, top_n=5):
    if query_word not in w2v_model.wv:
        return []
    return [(w, round(float(s), 4)) for w, s in w2v_model.wv.most_similar(query_word, topn=top_n)]


In [ ]:
n_vec_docs = 3000
vec_docs_raw = train_texts[:n_vec_docs]
vec_docs_tokens = [remove_stopwords(case_fold(tokenize(t))) for t in vec_docs_raw]
vec_docs_tokens = [d for d in vec_docs_tokens if len(d) > 0]
vec_docs_joined = [' '.join(d) for d in vec_docs_tokens]

X_termdoc, termdoc_terms = term_document_matrix(vec_docs_joined)
X_tfidf, tfidf_terms = tfidf_matrix(vec_docs_joined)
cooc, cooc_vocab = word_word_cooccurrence(vec_docs_tokens, window=CONTEXT_WINDOW)
ppmi = compute_ppmi(cooc)
print('term-doc matrix:', X_termdoc.shape, '| tfidf matrix:', X_tfidf.shape,
      '| co-occurrence matrix:', cooc.shape)


In [ ]:
w2v_model = train_word2vec(vec_docs_tokens, vector_size=100, window=CONTEXT_WINDOW,
                            min_count=3, seed=RANDOM_SEED)
print('Word2Vec vocab size:', len(w2v_model.wv))


In [ ]:
query_words = ['good', 'bad', 'love', 'happy', 'sad']
comparison_rows = []
for q in query_words:
    tfidf_nn = nearest_neighbours_tfidf(q, tfidf_terms, X_tfidf.T, top_n=5)
    ppmi_nn = nearest_neighbours_ppmi(q, cooc_vocab, ppmi, top_n=5)
    w2v_nn = nearest_neighbours_word2vec(q, w2v_model, top_n=5)
    comparison_rows.append({'query': q, 'TF-IDF NN': tfidf_nn, 'PPMI NN': ppmi_nn, 'Word2Vec NN': w2v_nn})

comparison_table = pd.DataFrame(comparison_rows)
comparison_table


**Discussion (fill in after inspecting the table above):** state which
representation's neighbours are most semantically coherent for your data —
typically Word2Vec captures paradigmatic similarity (synonyms/antonyms mixed
by sentiment polarity) best on short, noisy tweet text, while PPMI is more
sensitive to sparse co-occurrence noise and TF-IDF term-document similarity
mostly reflects topical/document overlap rather than word meaning.

## Save all output tables for the report

In [ ]:
summary_table.to_csv('outputs/stage1_split_summary.csv', index=False)
pd.DataFrame(examples).to_csv('outputs/stage2_stem_vs_lemma.csv', index=False)
pd.Series(report_vocab).to_csv('outputs/stage2_vocab_report.csv')
ppl_table.to_csv('outputs/stage3_perplexity.csv', index=False)
pd.DataFrame(top5_confused, columns=['(gold, predicted)', 'count']).to_csv('outputs/stage4_confusion.csv', index=False)
dep_table.to_csv('outputs/stage5_dependencies.csv', index=False)
pd.DataFrame(triples, columns=['subject', 'verb', 'object']).to_csv('outputs/stage5_svo_triples.csv', index=False)
pd.DataFrame(lesk_eval['details']).to_csv('outputs/stage6_lesk_eval.csv', index=False)
comparison_table.to_csv('outputs/stage7_vector_comparison.csv', index=False)
print('All tables saved to ./outputs/')

# Optional: zip everything for download
!zip -rq outputs.zip outputs
from google.colab import files
files.download('outputs.zip')
